In [1]:
import yaml
from json_repair import repair_json

from tool_description_optimizer.src.summary.state import *
from openai import OpenAI
from tool_description_optimizer.src.summary.llm_client import LLMClient
from typing import Any, Dict, Optional, Literal

# 相关配置
from tool_description_optimizer.src.summary.flow_config import FlowConfig
from tool_description_optimizer.common.flow.prompt_registry import PromptRegistry

# 相关执行class
from tool_description_optimizer.src.passk.nn_recall_passk import recall_passk_function

try:
    from langgraph.checkpoint.memory import InMemorySaver
    from langgraph.graph import END, START, StateGraph
    from langgraph.graph.message import add_messages
except ImportError as exc:  # pragma: no cover
    raise RuntimeError(
        "Missing langgraph dependencies. Please run: "
        "pip install langgraph langchain-core"
    ) from exc

from tool_description_optimizer.src.summary.llm_summary_optimizer import LLMDescriptionSummary

/root/paddlejob/workspace/env_run/output/zacharychu/miniconda3/envs/server/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:09<00:00, 43.50it/s]


In [6]:
from tool_description_optimizer.src.summary.llm_description_judge import LLMDescriptionJudge
class ToolOptimizerGraph:
    def __init__(
        self,
        resource_id: str,
        llm_client: Optional[LLMClient] = None,
        prompt_path: str = "../config/prompts.json",
        flow_config_path: str = "../config/agent_config.yaml",
        tools_description_path: str = "../data/summary/tool_descriptions.json",
        test_data_path: str = "../data/summary/query.json",
        checkpointer: Optional[Any] = None,
    ) -> None:
        # 设置通用的 参数
        self.prompts = PromptRegistry(prompt_path).get_promts()
        self.flow_config = FlowConfig(flow_config_path)
        self.checkpointer = checkpointer or InMemorySaver()
        self.last_tools_description_path = tools_description_path
        # 设置 tool resource_id
        self.resource_id = resource_id

        # 加载tools 这个列表
        self.tools_dict = load_tools_from_json(self.last_tools_description_path)

        # 加载测试集
        # test_data_path: str = "../data/summary/query.json"
        # load_tools_from_json(test_data_path)#
        self.query_good_all_dict = load_tools_from_json(test_data_path)
        self.query_good_dict = self.query_good_all_dict[resource_id]

        # node -> model
        self.node_model_map: Dict[str, str] = {} 
        if self.flow_config.node_model_map:
            self.node_model_map.update({str(k): str(v) for k, v in self.flow_config.node_model_map.items()})

        # node -> temperature
        self.node_temperature_map: Dict[str, float] =  {} 
        if self.flow_config.node_temperature_map:
            self.node_temperature_map.update({str(k): float(v) for k, v in self.flow_config.node_temperature_map.items()})
 
        llm_dict = {}
        if len(self.flow_config.clients) != 0:
            for k, v in self.flow_config.clients.items():
                if "tianchi" in v["base_url"]:
                    v['host'] = "tianchi-proxy.baidu-int.com"
                    v['appid'] = 'app-CdjpA4YQ'
                    llm_dict[k] = LLMClient(**v)

        # llm_client
        if llm_client:
            self.llm_client = llm_client
        else:
            self.llm_client = LLMClient(**self.flow_config.config["default_client"])

        # node_llm_client_map
        self.node_llm_clients: Dict[str, LLMClient] = {}
        if self.flow_config.node_llm_client_map:
            self.node_llm_clients.update({
                str(k): llm_dict[v] for k, v in self.flow_config.node_llm_client_map.items()
            })

    # ---------- generate text ----------

    # node -> client
    def _client_for(self, node_name: str) -> LLMClient:
        return self.node_llm_clients.get(node_name, self.llm_client)

    # node -> model name
    def _model_for(self, node_name: str) -> str:
        return self.node_model_map.get(node_name, "deepseek-v3.2")
    # node -> temperature 
    def _temperature_for(self, node_name: str) -> float:
        return self.node_temperature_map.get(node_name, 0.8)

    def _generate_text(
        self,
        node_name: str,
        prompt: str,
        max_tokens: int = 2048,
        extra_body: Dict[str, Any] = {}
    ) -> Any:
        client = self._client_for(node_name)
        response =  client.generate_text(
            prompt = prompt,
            model=self._model_for(node_name),
            temperature=self._temperature_for(node_name),
            extra_body = extra_body
        )
        return client.parse_chat_content(response)
    
    # --------- 统计和check工具 ----------------------
    def statistic_check_tool(self, 
                    state: ToolOptimizerState):
        """示例主函数：你可以替换为自己的 JSON 文件路径。"""
        best_record = state.get("best_record", None)
        if best_record is not None:
            self.last_tools_description_path = best_record.tool_path

        self.tools_dict = load_tools_from_json(self.last_tools_description_path) 
        statistic_check_version = "version_" + str(state.get("current_version_id", 0))
        print("当前执行的版本：", statistic_check_version)
        statistic_output = "../recall_outputs/summary/" + statistic_check_version + "/" + self.resource_id
    
        detaildf = recall_passk_function(self.tools_dict, self.query_good_dict, self.resource_id, statistic_output)

        # 计算 top 1 的 precision
        detaildf_top1 = detaildf[(detaildf['k']==1) & (detaildf['view']=="merged")]
        relevants = detaildf_top1['gold_ids'].to_list()
        retrieveds = detaildf_top1['recall_ids'].to_list()
        precision1 = precision_at_k_batch(retrieveds, relevants, 1)
        recall1 = recall_at_k_batch(retrieveds, relevants, 1)

        # 计算 top 3 的 precision
        detaildf_top3 = detaildf[(detaildf['k']==3) & (detaildf['view']=="merged")]
        relevants = detaildf_top3['gold_ids'].to_list()
        retrieveds = detaildf_top3['recall_ids'].to_list()
        precision3 = precision_at_k_batch(retrieveds, relevants, 3)
        recall3 = recall_at_k_batch(retrieveds, relevants, 3)

        # 统计 top 3 的结果
        tools_query = {}
        tools_case_ids = []
        for indx, row in detaildf_top3.iterrows():
            recall_ids = row["recall_ids"]
            if len(recall_ids) == 0:
                recall_ids = [NO_CALL] 
            if len(tools_query.get(recall_ids[0], [])) == 0:
                tools_query[recall_ids[0]] =  []
            tools_query[recall_ids[0]].append(row["query"])
            tools_case_ids.append(recall_ids[0])
        top_tools_case = {k: tools_query[k] for k in get_k_tool(tools_case_ids, 3)}
        tools_case_all = {k: tools_query[k] for k in set(tools_case_ids)}

        version_id, next_version_id = next_version_pair(state)

        versionInfo = VersionRecord(
            version_id = statistic_check_version, 
            parent_version_id = version_id,
            stage = "statistic",
            description = self.tools_dict[self.resource_id]["description"],
            case_result = tools_case_all,
            top_case = top_tools_case,
            tool_path = statistic_output + "/tool_prompt.json",
            recall1 = recall1,
            precision1 = precision1,
            recall3 = recall3,
            precision3 = precision3
        )

        # 判断是否需要更新最佳记录
        # 条件1: best_record 不存在 (即为 None)
        # 条件2: 新的指标 (recall3, precision3) 优于旧的 best_record
        should_update = (best_record is None) or \
                (recall3 > best_record.recall3 and precision3 > best_record.precision3)

        with open(statistic_output + "/tool_prompt.json","w") as w:
            json.dump(self.tools_dict, w, ensure_ascii=False, indent=2)

        if should_update:
            return {
                    "current_version_id": version_id,
                    "next_version_id": next_version_id,
                    "version_history": append_version_history(state, versionInfo),
                    "best_description": state.get("current_description", ""),
                    "best_version_id": state.get("current_version_id", 0),
                    "best_record": versionInfo,
                    "inner_loop_cnt":0

                }
        else:
            return {
                "current_version_id": version_id,
                "next_version_id": next_version_id,
                "version_history": append_version_history(state, versionInfo),
                "inner_loop_cnt": 0
            }

    # ---------  LLMNodeCritic node--------------
    def llm_description_judge(self,  state: ToolOptimizerState):
        prompt_and_desc = LLMDescriptionJudge._gen_prompt(state, self.prompts['judge'])
        if not prompt_and_desc:
            print(f"[judge] optimizer_history 为空，跳过本轮评审")
            version_id = state.get("next_version_id", 1)
            return {
                "current_version_id": version_id,
                "next_version_id": version_id + 1,
            }
        prompt, optimizer_description = prompt_and_desc

        for _ in range(3):
            response, stype, flag = self._generate_text(node_name = "judge", prompt = prompt)
            response, flag, error_type = LLMDescriptionJudge._vertify_result(response)
            if flag:
                break
        if flag:
            relevance_score = response["relevance_score"]
            if relevance_score == 3 or  relevance_score == 2:
                self.tools_dict[self.resource_id]['description'] = optimizer_description

            judge_record = InfoRecord(
                version_id = state.get("current_version_id", "-1"),
                stage = "judge",
                info = response
            )
            return {
                "inner_loop_cnt": state.get("inner_loop_cnt", 3) + 1,
                "judge_history": append_judge_history(state, judge_record)
            }
        else:
            {
                "inner_loop_cnt": state.get("inner_loop_cnt", 3) + 1
            }
    # ---------  LLMNodeSummary node--------------
    def llm_description_summary(self, 
                                  state: ToolOptimizerState):
        prompt = LLMDescriptionSummary._gen_prompt(state, self.prompts['summary'], list(self.query_good_dict.keys()))

        for _ in range(3):
            response, stype, flag = dag._generate_text(node_name = "summary", prompt = prompt)
            response, flag, error_type = LLMDescriptionSummary._vertify_result(response)
            if flag:
                break
        if flag:
            optimizer_description = response["optimizer_description"]
            
            
            print("description: ", state.get("current_version_id", "-1"), self.tools_dict[self.resource_id]['description'])
            optimizer_record = InfoRecord(
                version_id = state.get("current_version_id", "-1"),
                stage = "summary",
                info = response
            )
            return {
                "optimizer_history": append_optimizer_history(state, optimizer_record)
            }


    # ---------- graph build ----------

def build_optimizer_graph():
    """Build and compile the LangGraph optimization workflow."""

    graph = StateGraph(ToolOptimizerState)
    graph.add_node("optimizer", optimizer)
    graph.add_node("vertify_after_optimizer", vertify_after_optimizer)
    graph.add_edge("bump_iteration", "optimizer")
    return graph.compile()


def should_continue(state: ToolOptimizerState) -> Literal["summary_optimizer", END]:
    # 达到最大轮次
    max_iteration = state.get("max_iterations", 3)
    if state.get("current_version_id", 10) > max_iteration:
        return END
    return "summary_optimizer"

# 第二层路由：judge节点结束后，控制内层循环3次
def summary_loop_router(state: ToolOptimizerState):
    if state["inner_loop_cnt"] < 3:
        return "summary_optimizer"
    # 已满3次：内层循环结束，退回统计节点做效果核验
    return "statistic_check_tool"

#------- init ----
def list_file(folder_path):
    all_items = os.listdir(folder_path)
    # 只列出文件（不包括文件夹）
    files_only = [os.path.join(folder_path, item) for item in all_items if os.path.isfile(os.path.join(folder_path, item)) and item.endswith(".pkl")]
    print("\n只列出文件:")
    return files_only
resource_ids_all = []
for path in list_file("../recall_outputs/summary/"):
    resource_ids_all.append(path.split("/")[-1].replace(".pkl", ""))



只列出文件:


In [12]:
query_good_all_dict = load_tools_from_json("../data/summary/query.json")
print(query_good_all_dict.keys())
query_good_all_dict["61277"]

dict_keys(['5103', '5359', '36607', '46671', '50159', '50160', '50161', '50162', '50354', '50709', '50926', '51088', '51255', '51590', '5293', '5351', '5428', '5432', '5802', '23', '50143', '62172', '85', '50133', '50192', '5671', '60868', '60872', '60874', '62106', '62306', '5851', '4526', '50574', '50760', '5271', '5819', '50393', '5179', '62248', '62206', '8470', '51323', '62016', '51275', '51474', '5544', '60338', '60540', '60977', '61800', '5881', '5882', '46679', '47190', '47198', '47242', '47266', '47272', '5012', '5778', '5273', '5917', '60293', '51712', '267', '5143', '5144', '39529', '39537', '4551', '52364', '51129', '5550', '4999', '51044', '5601', '61111', '61277', '4982', '51482', '51376', '51377', '51378', '51379', '51380', '51381', '51382', '51386', '51387', '51388', '52336', '52523', '52524', '28126', '28128', '50731', '51457', '62497', '60265'])


{'昆明市7月份天气情况': ['61277'],
 '天气预报30天查询30': ['61277'],
 '2025年5月份天气预报': ['61277'],
 '7月份天气预报': ['61277'],
 '四十天天气预报': ['61277'],
 '保定未来30天天气情况': ['61277'],
 '未来15天天气预报查询度#': ['61277'],
 '宁波六月份天气预报': ['61277'],
 '监利市三十天天气预报': ['61277'],
 '岳阳40天天气预报查询': ['61277'],
 '青岛未来15天天气预报': ['61277'],
 '北京天气25号到30号': ['61277'],
 '上海一个月内天气预报': ['61277'],
 '重庆天气预报一个月30天查询结果': ['61277'],
 '石家庄40天天气预报': ['61277'],
 '成都30天天气预报查询': ['61277'],
 '莱西2026年07月天气': ['61277'],
 '南昌未来60天天气预报查询': ['61277'],
 '黑龙江伊春30天天气预报': ['61277'],
 '秦皇岛未来15天的天气预报情况': ['61277'],
 '嘉兴最近一个月天气情况怎么样': ['61277'],
 '7月4日天气': ['61277'],
 '眉山未来一个月的天气预报': ['61277'],
 '平顶山未来40天的天气预报': ['61277'],
 '湘西天气40天查询结果': ['61277'],
 '漯河市40天天气预报': ['61277'],
 '赤峰市未来15天天气预报': ['61277'],
 '8月份天气': ['61277'],
 '北海7月份天气预报查询': ['61277'],
 '泗县三十天天气预报': ['61277'],
 '廊坊天气预报2025年': ['61277'],
 '焦作三十天天气预报': ['61277'],
 '毕节40天内天气预报': ['61277'],
 '咸宁40天天气预报': ['61277'],
 '武汉市7月份天气情况': ['61277'],
 '杭州市7月份天气预报': ['61277'],
 '这20天的天气预报': ['61277'],
 '宜宾8月份天气预报': [

In [7]:
state = ""
for resource_id, value in query_good_all_dict.items():
    if resource_id in resource_ids_all or resource_id in ["47242"]:
        print(resource_id, "skip!!!")
        continue
    if resource_id != '61277':
        continue
    print("开始执行：", resource_id)
    try:
        dag = ToolOptimizerGraph(resource_id = resource_id,
                                 prompt_path ="../../config/prompts.json",
                                 flow_config_path ="../../config/agent_config.yaml",
                                 tools_description_path ="../../data/data_import/summary/tool_description.json",
                                 test_data_path  ="../../data/data_import/summary/query.json",
                                 checkpointer = None)

        graph = StateGraph(ToolOptimizerState)
        graph.add_node("statistic_check_tool", dag.statistic_check_tool)
        graph.add_node("summary_optimizer", dag.llm_description_summary)
        graph.add_node("description_judge", dag.llm_description_judge)
        graph.set_entry_point("statistic_check_tool")
        # graph.add_edge("statistic_check_tool", END)
        graph.add_conditional_edges(
            "statistic_check_tool",
            should_continue,
            {
                "summary_optimizer": "summary_optimizer",
                END: END
            }
        )
        graph.add_edge("summary_optimizer", "description_judge")
        # 3. 评审节点走条件路由：未满3轮继续优化；满3轮回到统计节点
        graph.add_conditional_edges(
            source="description_judge",
            path=summary_loop_router,
            path_map={
                "summary_optimizer": "summary_optimizer",
                "statistic_check_tool": "statistic_check_tool"
            }
        )
        compiled_graph = graph.compile()

        initial_state = {
                "resource_id": resource_id,
                "title": dag.tools_dict[resource_id]["title"],
                "original_description": dag.tools_dict[resource_id]["description"],
                # 当前工作基线指针（可手动/自动回滚）
                "current_version_id": 0,
                "current_description": dag.tools_dict[resource_id]["description"],
                # 下一个版本的id
                "next_version_id": 1,
                # 固定参数
                "max_iterations": 3,
                "iteration": 0,
                # 版本历史仓库：全量快照存储
                "version_history": [],
                "judge_history":[],
                #记录历史
                "optimizer_history": []
            }
        state = compiled_graph.invoke(initial_state)  
    except:
        continue

NameError: name 'query_good_all_dict' is not defined

In [8]:
resource_id = '61277'



当前执行的版本： version_0
开始建库: description
Building embeddings for view=description, size=100
开始建库: title_description
Building embeddings for view=title_description, size=100
Total tools indexed: 100
Total eval queries: 133
召回结果保存完成：
summary_csv: ../recall_outputs/summary/version_0/61277/tool_pass_at_k_recall_summary.csv
detail_csv: ../recall_outputs/summary/version_0/61277/tool_pass_at_k_recall_detail.csv
summary_jsonl: ../recall_outputs/summary/version_0/61277/tool_pass_at_k_recall_summary.jsonl
detail_jsonl: ../recall_outputs/summary/version_0/61277/tool_pass_at_k_recall_detail.jsonl
excel: ../recall_outputs/summary/version_0/61277/tool_pass_at_k_recall.xlsx
description:  1 该工具提供该工具提供目标城市指定月份的逐日气象数据，支持日历与列表两种展示模式，直观呈现每日天气图标、当日最高及最低气温，并高亮标记当前日期。工具支持灵活切换城市与月份，便于用户查看不同地区、不同时段的天气走势；同时提供展开更多日期信息及跳转至长期天气趋势页面的扩展功能。适用场景包括月度出行规划、旅行日程安排、日常穿搭参考等，可高效响应用户对某地多日天气、逐日气温变化及未来气候趋势的查询需求，辅助用户提前制定活动计划。
description:  1 该工具提供指定城市未来多日（如30天、40天、60天或整月）的逐日天气预报，包括每日最高最低气温、天气图标，支持日历与列表展示。用户可自由切换城市和查询月份，快速查看不同地区长期天气走

In [30]:
def append_judge_history(state: ToolOptimizerState, record: InfoRecord) -> list[InfoRecord]:
    """追加版本快照至历史仓库，不修改原有记录"""
    return [*state["judge_history"], record]

In [9]:
state

{'title': '多天天气新（历史+未来）',
 'original_description': '该工具提供该工具提供目标城市指定月份的逐日气象数据，支持日历与列表两种展示模式，直观呈现每日天气图标、当日最高及最低气温，并高亮标记当前日期。工具支持灵活切换城市与月份，便于用户查看不同地区、不同时段的天气走势；同时提供展开更多日期信息及跳转至长期天气趋势页面的扩展功能。适用场景包括月度出行规划、旅行日程安排、日常穿搭参考等，可高效响应用户对某地多日天气、逐日气温变化及未来气候趋势的查询需求，辅助用户提前制定活动计划。',
 'resource_id': '61277',
 'current_description': '该工具提供该工具提供目标城市指定月份的逐日气象数据，支持日历与列表两种展示模式，直观呈现每日天气图标、当日最高及最低气温，并高亮标记当前日期。工具支持灵活切换城市与月份，便于用户查看不同地区、不同时段的天气走势；同时提供展开更多日期信息及跳转至长期天气趋势页面的扩展功能。适用场景包括月度出行规划、旅行日程安排、日常穿搭参考等，可高效响应用户对某地多日天气、逐日气温变化及未来气候趋势的查询需求，辅助用户提前制定活动计划。',
 'current_version_id': 4,
 'next_version_id': 5,
 'inner_loop_cnt': 0,
 'best_description': '该工具提供该工具提供目标城市指定月份的逐日气象数据，支持日历与列表两种展示模式，直观呈现每日天气图标、当日最高及最低气温，并高亮标记当前日期。工具支持灵活切换城市与月份，便于用户查看不同地区、不同时段的天气走势；同时提供展开更多日期信息及跳转至长期天气趋势页面的扩展功能。适用场景包括月度出行规划、旅行日程安排、日常穿搭参考等，可高效响应用户对某地多日天气、逐日气温变化及未来气候趋势的查询需求，辅助用户提前制定活动计划。',
 'best_version_id': 0,
 'best_record': VersionRecord(version_id='version_0', stage='statistic', description='该工具提供该工具提供目标城市指定月份的逐日气象数据，支持日历与列表两种展示模式，直观呈现每日天气图标、当日最

In [4]:
# from tool_summary_main import load_tools_from_json
query_good_all_dict = load_tools_from_json("../../data/data_not_import/summary/query.json")
for resource_id, value in query_good_all_dict.items():
    print(resource_id, value)

10102 {'专利业务办理系统': ['10102'], '德高防水': ['10102'], '东本贴吧': ['10102'], '拼多多拼多多': ['10102'], '国家药品监督管理局': ['10102'], '中国驻英国大使馆': ['10102'], '吉林省前卫医院': ['10102'], 'mc.': ['10102'], 'yige': ['10102'], '外婆味道': ['10102'], '股市通': ['10102'], 'claude': ['10102'], '原麦山丘': ['10102'], '小狐狸钱包': ['10102'], '武炼顶峰': ['10102'], 'chatgpt网址': ['10102'], '金域医学': ['10102'], 'www.24': ['10102'], '百度百家账号注册入口': ['10102'], '土流网': ['10102'], '携程租车': ['10102'], '百度文心助手': ['10102'], '诡秘之主第二季': ['10102'], '文心一言4.5': ['10102'], '格林豪泰酒店': ['10102'], 'id3': ['10102'], '华润集团': ['10102'], '四川银行招聘官网': ['10102'], '中国旅游文化资源开发促进会': ['10102'], '广东工商学院': ['10102'], '蛋仔 派对': ['10102'], '宿命之环': ['10102'], 'tensorflow': ['10102'], 'robotaxi': ['10102'], '大连外国语大学': ['10102'], '马来西亚电子入境卡': ['10102'], '郑州交通技师学院': ['10102'], '两江新区人民政府': ['10102'], '元宝官网': ['10102'], '中国联通': ['10102'], '即梦ai官网': ['10102'], '华东交通大学': ['10102'], '极越汽车': ['10102'], '山中大学': ['10102'], '欧易下载': ['10102'], '极客湾': ['10102'], '神秘复苏。': ['10102'], '度加': ['10102'

In [3]:
def load_tools_from_json(path: str):
    """读取 tool 集合 JSON 文件。"""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data